# 04. FragPipe search: workflow, manifests and launchers

Writes everything FragPipe needs to run the re-search without the graphical interface:

- `config/MOSAIC_TMT10_variants.workflow`: the stock `TMT10` workflow of FragPipe 24.0 with the database set and the pooled channel declared as the reference (as in the stock `TMT10-bridge` workflow);
- `config/SET_n.fp-manifest` and `config/ALL_SETS.fp-manifest`: lists of raw files per TMT set;
- `RUN_SEARCH_SET_n.bat` and `RUN_SEARCH_ALL_SETS.bat`: launchers for Windows.

Acquisition as reported by Kim et al. 2022: Q-Exactive, HCD MS2 at 35,000 resolution (reporter ions quantified at MS2), trypsin/Lys-C, iodoacetamide, TMT 10-plex, 24 high-pH fractions per set.
The stock workflow matches this: strict trypsin with up to 2 missed cleavages, 20 ppm precursor and fragment tolerance with mass calibration, carbamidomethyl-Cys and TMT-Lys fixed, Met oxidation, protein N-terminal acetylation and peptide N-terminal TMT variable, MSBooster rescoring, Percolator, 1% protein FDR, IonQuant reporter extraction and TMT-Integrator summarisation.

In [1]:
from pathlib import Path, PureWindowsPath
import os, re
import pandas as pd

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
CFG = ROOT / "config"

# Windows locations used inside the generated files
# Project location on the Windows search machine: detected automatically when this notebook runs on Windows;
# otherwise set the environment variable MOSAIC_WIN_ROOT (the default is the location used for the published run).
WIN_ROOT = PureWindowsPath(os.environ.get("MOSAIC_WIN_ROOT", str(ROOT) if os.name == "nt" else r"C:\MOSAIC_reanaliza_PXD029845"))
FRAGPIPE = PureWindowsPath(os.environ.get("MOSAIC_FRAGPIPE", r"C:\FragPipe\FragPipe-24.0"))
RAM_GB, THREADS = 24, 18            # machine: 32 GB RAM, 20 logical cores
assert " " not in str(WIN_ROOT), "paths with spaces break some FragPipe tools"

## Workflow file

In [2]:
def prop(path):                      # Java .properties escaping of a Windows path
    return str(path).replace("\\", "\\\\").replace(":", "\\:")

changes = {
    "database.db-path": prop(WIN_ROOT / "database" / "MOSAIC_human_sp_variants_contam_decoys.fasta"),
    "database.decoy-tag": "rev_",
    "tmtintegrator.ref_tag": "pool",          # channel 131 is named pool_SETn in the annotation files
    "tmtintegrator.add_Ref": "-1",            # a real reference channel is present
}
lines = (CFG / "TMT10_FragPipe24_original.workflow").read_text(encoding="utf-8").splitlines()
seen, out = set(), []
for line in lines:
    key = line.split("=", 1)[0]
    if key in changes:
        out.append(f"{key}={changes[key]}"); seen.add(key)
    else:
        out.append(line)
out += [f"{k}={v}" for k, v in changes.items() if k not in seen]
(CFG / "MOSAIC_TMT10_variants.workflow").write_text("\n".join(out) + "\n", encoding="utf-8")

orig = dict(l.split("=", 1) for l in lines if "=" in l and not l.startswith("#"))
pd.DataFrame([(k, orig.get(k, "(absent)"), v) for k, v in changes.items()], columns=["key", "stock TMT10", "this analysis"])

,key,stock TMT10,this analysis
0,database.db-path,(absent),C\:\\MOSAIC_reanaliza_PXD029845\\database\\MOS...
1,database.decoy-tag,rev_,rev_
2,tmtintegrator.ref_tag,Bridge,pool
3,tmtintegrator.add_Ref,1,-1


## Channel annotation files

Written from the sample map recovered in notebook 02. FragPipe 24 names the last TMT-10 channel `131N` (Proteome Discoverer calls it `131`); the pooled reference is in that channel in every set.

In [3]:
sample_map = pd.read_csv(ROOT / "results" / "sample_map.tsv", sep="\t", dtype={"channel": str})
TMT10 = ["126", "127N", "127C", "128N", "128C", "129N", "129C", "130N", "130C", "131N"]
for s in range(1, 5):
    by_channel = sample_map[sample_map["tmt_set"] == s].set_index("channel")["sample"].to_dict()
    lines = [f"{ch} {by_channel.get(ch, by_channel.get(ch.rstrip('N'), f'pool_SET{s}'))}" for ch in TMT10]
    assert sum("pool" in l for l in lines) == 1 and lines[-1].endswith(f"pool_SET{s}")
    (ROOT / "data_raw" / f"SET_{s}" / f"SET_{s}_annotation.txt").write_text("\n".join(lines) + "\n", encoding="ascii", newline="\n")
    print(f"SET_{s}:", "; ".join(lines))

SET_1: 126 S1; 127N S2; 127C C1; 128N S3; 128C S4; 129N C2; 129C S5; 130N C3; 130C S6; 131N pool_SET1
SET_2: 126 S7; 127N C4; 127C S8; 128N S9; 128C C5; 129N S10; 129C S11; 130N S12; 130C C6; 131N pool_SET2
SET_3: 126 C7; 127N S13; 127C S14; 128N C8; 128C S15; 129N S16; 129C C9; 130N S17; 130C S18; 131N pool_SET3
SET_4: 126 C10; 127N S19; 127C C11; 128N S20; 128C S21; 129N C12; 129C S22; 130N S23; 130C C13; 131N pool_SET4


## Manifests

One experiment per TMT set. FragPipe finds the channel annotation (`SET_n_annotation.txt`) in the folder of each set.

In [4]:
rows = []
for s in range(1, 5):
    files = sorted((ROOT / "data_raw" / f"SET_{s}").glob("*.raw"), key=lambda p: int(re.search(r"frxn_(\d+)", p.name).group(1)))
    assert len(files) == 24, (s, len(files))
    part = [(str(WIN_ROOT / "data_raw" / f"SET_{s}" / f.name), f"SET_{s}", "", "DDA") for f in files]
    pd.DataFrame(part).to_csv(CFG / f"SET_{s}.fp-manifest", sep="\t", header=False, index=False, lineterminator="\n")
    rows += part
pd.DataFrame(rows).to_csv(CFG / "ALL_SETS.fp-manifest", sep="\t", header=False, index=False, lineterminator="\n")
print(len(rows), "raw files in", len({r[1] for r in rows}), "experiments")
for s in range(1, 5):
    print(f"SET_{s}:", (ROOT / "data_raw" / f"SET_{s}" / f"SET_{s}_annotation.txt").read_text().replace("\n", "; "))

96 raw files in 4 experiments
SET_1: 126 S1; 127N S2; 127C C1; 128N S3; 128C S4; 129N C2; 129C S5; 130N C3; 130C S6; 131N pool_SET1; 
SET_2: 126 S7; 127N C4; 127C S8; 128N S9; 128C C5; 129N S10; 129C S11; 130N S12; 130C C6; 131N pool_SET2; 
SET_3: 126 C7; 127N S13; 127C S14; 128N C8; 128C S15; 129N S16; 129C C9; 130N S17; 130C S18; 131N pool_SET3; 
SET_4: 126 C10; 127N S19; 127C C11; 128N S20; 128C S21; 129N C12; 129C S22; 130N S23; 130C C13; 131N pool_SET4; 


## Launchers

In [5]:
BAT = """@echo off
setlocal
cd /d "%~dp0"
if not defined FRAGPIPE_HOME set FRAGPIPE_HOME={fp}
set FP=%FRAGPIPE_HOME%
set JAVA_HOME=%FP%\\jre
set OUT=results_large\\search_{tag}
if not exist "%OUT%" mkdir "%OUT%"
echo Start: %date% %time% > "%OUT%\\run_times.txt"
echo FragPipe search {tag} is running. Progress is written to %OUT%\\run_log.txt
echo Do not close this window.
call "%FP%\\bin\\fragpipe.bat" --headless --workflow "%~dp0config\\MOSAIC_TMT10_variants.workflow" --manifest "%~dp0config\\{tag}.fp-manifest" --workdir "%~dp0%OUT%" --config-tools-folder "%FP%\\tools" --ram {ram} --threads {thr} > "%OUT%\\run_log.txt" 2>&1
echo End: %date% %time% >> "%OUT%\\run_times.txt"
echo.
echo FINISHED. See %OUT%\\run_log.txt
pause
"""
for tag in ["SET_1", "SET_2", "SET_3", "SET_4", "ALL_SETS"]:
    text = BAT.format(fp=FRAGPIPE, tag=tag, ram=RAM_GB, thr=THREADS)
    (ROOT / f"RUN_SEARCH_{tag}.bat").write_text(text.replace("\n", "\r\n"), encoding="ascii", newline="")
print((ROOT / "RUN_SEARCH_SET_1.bat").read_text())

@echo off
setlocal
cd /d "%~dp0"
if not defined FRAGPIPE_HOME set FRAGPIPE_HOME=C:\FragPipe\FragPipe-24.0
set FP=%FRAGPIPE_HOME%
set JAVA_HOME=%FP%\jre
set OUT=results_large\search_SET_1
if not exist "%OUT%" mkdir "%OUT%"
echo Start: %date% %time% > "%OUT%\run_times.txt"
echo FragPipe search SET_1 is running. Progress is written to %OUT%\run_log.txt
echo Do not close this window.
call "%FP%\bin\fragpipe.bat" --headless --workflow "%~dp0config\MOSAIC_TMT10_variants.workflow" --manifest "%~dp0config\SET_1.fp-manifest" --workdir "%~dp0%OUT%" --config-tools-folder "%FP%\tools" --ram 24 --threads 18 > "%OUT%\run_log.txt" 2>&1
echo End: %date% %time% >> "%OUT%\run_times.txt"
echo.
echo FINISHED. See %OUT%\run_log.txt
pause



## Order of runs

1. `RUN_SEARCH_SET_1.bat`: one TMT set (24 files). Gives the run time and the first variant peptides.
2. `RUN_SEARCH_ALL_SETS.bat`: all four sets in one analysis, so that TMT-Integrator can combine them through the pooled reference.

## Quantification-only rerun

If identification has finished but reporter-ion quantification has to be repeated (for example after correcting an annotation file), this workflow skips the search, rescoring and protein inference and reruns IonQuant and TMT-Integrator on the existing `psm.tsv` files.

In [6]:
quant_only = {"msfragger.run-msfragger": "false", "msbooster.run-msbooster": "false", "percolator.run-percolator": "false",
              "protein-prophet.run-protein-prophet": "false", "phi-report.run-report": "false", "run-psm-validation": "false"}
lines = (CFG / "MOSAIC_TMT10_variants.workflow").read_text(encoding="utf-8").splitlines()
out = [f"{l.split('=', 1)[0]}={quant_only[l.split('=', 1)[0]]}" if l.split("=", 1)[0] in quant_only else l for l in lines]
(CFG / "MOSAIC_TMT10_variants_quant_only.workflow").write_text("\n".join(out) + "\n", encoding="utf-8")
text = (BAT.format(fp=FRAGPIPE, tag="ALL_SETS", ram=RAM_GB, thr=THREADS)
        .replace("MOSAIC_TMT10_variants.workflow", "MOSAIC_TMT10_variants_quant_only.workflow")
        .replace("run_log.txt", "run_log_quant_only.txt").replace("run_times.txt", "run_times_quant_only.txt")
        .replace("FragPipe search ALL_SETS", "FragPipe quantification (no search) ALL_SETS"))
(ROOT / "RUN_QUANT_ONLY_ALL_SETS.bat").write_text(text.replace("\n", "\r\n"), encoding="ascii", newline="")
print(text)

@echo off
setlocal
cd /d "%~dp0"
if not defined FRAGPIPE_HOME set FRAGPIPE_HOME=C:\FragPipe\FragPipe-24.0
set FP=%FRAGPIPE_HOME%
set JAVA_HOME=%FP%\jre
set OUT=results_large\search_ALL_SETS
if not exist "%OUT%" mkdir "%OUT%"
echo Start: %date% %time% > "%OUT%\run_times_quant_only.txt"
echo FragPipe quantification (no search) ALL_SETS is running. Progress is written to %OUT%\run_log_quant_only.txt
echo Do not close this window.
call "%FP%\bin\fragpipe.bat" --headless --workflow "%~dp0config\MOSAIC_TMT10_variants_quant_only.workflow" --manifest "%~dp0config\ALL_SETS.fp-manifest" --workdir "%~dp0%OUT%" --config-tools-folder "%FP%\tools" --ram 24 --threads 18 > "%OUT%\run_log_quant_only.txt" 2>&1
echo End: %date% %time% >> "%OUT%\run_times_quant_only.txt"
echo.
echo FINISHED. See %OUT%\run_log_quant_only.txt
pause

